### Credits:

<img align="left" src="https://ithaka-labs.s3.amazonaws.com/static-files/images/tdm/tdmdocs/CC_BY.png"><br />

This notebook is created by Zhuo Chen based on the notebooks created by [Nathan Kelber](http://nkelber.com), [William Mattingly](https://github.com/wjbmattingly/tap-2022-pandas) and [Melanie Walsh](https://github.com/melaniewalsh/Data-Analysis-with-Pandas) under [Creative Commons CC BY License](https://creativecommons.org/licenses/by/4.0/).<br />
For questions/comments/improvements, email zhuo.chen@ithaka.org or nathan.kelber@ithaka.org<br />

Reused and modified for internal use at Università Cattolica del Sacro Cuore di Milano, by Deborah Grbac, email deborah.grbac@unicatt.it and Valentina Schiariti, email valentina.schiariti-collaboratore@unicatt.it, released under CC BY License.

This repository is founded on **Constellate notebooks**. The original Jupyter notebooks repository was designed by the educators at **ITHAKA's Constellate project**. The project was sunset on July 1, 2025. This current repository uses and resuses Constellate notebooks as Open Educational Resources (OER), free for re-use under a Creative Commons CC BY License.
___

# Pandas Basics 3 

**Description:** This notebook describes how to:

* Use `merge()` and `concat()` to combine multiple dataframes
* Handle duplicates and get unique values
* Understand vectorized operations in Pandas
* Use `apply()` and `where()` to process the data in a dataframe

This is the third notebook in a series on learning to use Pandas. 

___


In [1]:
# Import pandas library, `as pd` allows us to shorten typing `pandas` to `pd` when we call pandas
import pandas as pd

## Merge and concatenate dataframes 

In this section, we will learn two methods to combine multiple dataframes in Pandas. 

### merge()

When dealing with tabular data, we often find ourselves in a situation where we need to merge two or more dataframes as we want a subset of the data from each. Pandas provides a `.merge()` method that allows us to merge dataframes easily.

The data we use in this section is the unemployment data in Massachusetts from 2023 January to February.

In [55]:
from pathlib import Path

# Check if a data folder exists. If not, create it.
data_folder = Path('./data/')
data_folder.mkdir(exist_ok=True)

In [63]:
# create a dataframe from the csv files
df1 = pd.read_csv('./data/Pandas3_unemp_jan.csv')
df2 = pd.read_csv('./data/Pandas3_unemp_feb.csv')

In [67]:
# take a look at df1
df1

,COUNTY,Jan Unemployment
0,BARNSTABLE,7233
1,BERKSHIRE,3078
2,BRISTOL,15952
3,DUKES,696
4,ESSEX,17573
5,FRANKLIN,1506
6,HAMPDEN,11629
7,HAMPSHIRE,3354
8,MIDDLESEX,29445
9,NANTUCKET,949


In [66]:
# take a look at df2
df2

,COUNTY,Feb Unemployment
0,BARNSTABLE,7855
1,BERKSHIRE,3162
2,BRISTOL,17112
3,DUKES,772
4,ESSEX,18486
5,FRANKLIN,1557
6,HAMPDEN,11941
7,HAMPSHIRE,3342
8,MIDDLESEX,29901
9,NANTUCKET,1072


The two dataframes have a column in common, the `COUNTY` column. This column contains the names of the **14 counties** in the state of Massachusetts. 
We can merge the two dataset using the `merge( )` method.

In [68]:
# merge the two dfs
df1.merge(df2)

,COUNTY,Jan Unemployment,Feb Unemployment
0,BARNSTABLE,7233,7855
1,BERKSHIRE,3078,3162
2,BRISTOL,15952,17112
3,DUKES,696,772
4,ESSEX,17573,18486
5,FRANKLIN,1506,1557
6,HAMPDEN,11629,11941
7,HAMPSHIRE,3354,3342
8,MIDDLESEX,29445,29901
9,NANTUCKET,949,1072


By default, `merge()` uses the columns that have the same name in both dataframes. However, we can explicitly specify the column through which we want to merge by using the `on` parameter.

In [69]:
# merge the two dfs
df1.merge(df2, on='COUNTY')

,COUNTY,Jan Unemployment,Feb Unemployment
0,BARNSTABLE,7233,7855
1,BERKSHIRE,3078,3162
2,BRISTOL,15952,17112
3,DUKES,696,772
4,ESSEX,17573,18486
5,FRANKLIN,1506,1557
6,HAMPDEN,11629,11941
7,HAMPSHIRE,3354,3342
8,MIDDLESEX,29445,29901
9,NANTUCKET,949,1072


In the previous merge, pandas matches the rows using the values in the COUNTY column.

However, datasets do not always match perfectly. Some rows may be present in one dataframe but missing from the other. For this reason, pandas allows us to choose different types of merge using the `how` parameter.

### inner join

In [74]:
# see what happens when there are non-matching keys 
df1 = unemp_jan.drop(0)
df2 = unemp_feb.drop(13)

In an inner join, you'll lose the rows that do not have a matching key in the two dataframes. In other words, only rows from the two dataframes that have a matching value in the **key column** will appear after the merge. 

In [75]:
# set the parameter 'how' to specify the merge type
df1.merge(df2, on='COUNTY', how='inner')

,COUNTY,Jan Unemployment,Feb Unemployment
0,BERKSHIRE,3078,3162
1,BRISTOL,15952,17112
2,DUKES,696,772
3,ESSEX,17573,18486
4,FRANKLIN,1506,1557
5,HAMPDEN,11629,11941
6,HAMPSHIRE,3354,3342
7,MIDDLESEX,29445,29901
8,NANTUCKET,949,1072
9,NORFOLK,14193,14356


As you can see, we have lost the data for Barnstable and Worcester after the merge. This is because the former has been dropped from the left dataframe and the latter has been dropped from the right dataframe. 

By default, pandas uses an inner merge. This means that, if we do not specify `how`, pandas keeps only the rows where the value of `COUNTY` appears in both dataframes.

In [76]:
df1.merge(df2, on ="COUNTY")

,COUNTY,Jan Unemployment,Feb Unemployment
0,BERKSHIRE,3078,3162
1,BRISTOL,15952,17112
2,DUKES,696,772
3,ESSEX,17573,18486
4,FRANKLIN,1506,1557
5,HAMPDEN,11629,11941
6,HAMPSHIRE,3354,3342
7,MIDDLESEX,29445,29901
8,NANTUCKET,949,1072
9,NORFOLK,14193,14356


### outer join

There are other types of merge, of course. An outer merge keeps all the rows from both dataframes. The missing values are replaced by NaN.

In [77]:
# change the merge type and see what happens
df1.merge(df2, on="COUNTY", how="outer")

,COUNTY,Jan Unemployment,Feb Unemployment
0,BARNSTABLE,NaN,7855.0
1,BERKSHIRE,3078.0,3162.0
2,BRISTOL,15952.0,17112.0
3,DUKES,696.0,772.0
4,ESSEX,17573.0,18486.0
5,FRANKLIN,1506.0,1557.0
6,HAMPDEN,11629.0,11941.0
7,HAMPSHIRE,3354.0,3342.0
8,MIDDLESEX,29445.0,29901.0
9,NANTUCKET,949.0,1072.0


### left outer join

A **left merge** keeps all the rows from the first dataframe, `df1`. If some rows in df1 do not have a match in `df2`, they are still kept, but the columns from `df2` contain NaN.


In [79]:
# Use the keys from the left df in the merge
df1.merge(df2, on="COUNTY", how="left")

,COUNTY,Jan Unemployment,Feb Unemployment
0,BERKSHIRE,3078,3162.0
1,BRISTOL,15952,17112.0
2,DUKES,696,772.0
3,ESSEX,17573,18486.0
4,FRANKLIN,1506,1557.0
5,HAMPDEN,11629,11941.0
6,HAMPSHIRE,3354,3342.0
7,MIDDLESEX,29445,29901.0
8,NANTUCKET,949,1072.0
9,NORFOLK,14193,14356.0


### right outer join

A **right merge** keeps all the rows from the second dataframe, `df2`. If some rows in `df2` do not have a match in `df1`, they are still kept, but the columns from `df1` contain NaN.

In [80]:
# Use the keys from the right df in the merge
df1.merge(df2, on="COUNTY", how="right")

,COUNTY,Jan Unemployment,Feb Unemployment
0,BARNSTABLE,NaN,7855
1,BERKSHIRE,3078.0,3162
2,BRISTOL,15952.0,17112
3,DUKES,696.0,772
4,ESSEX,17573.0,18486
5,FRANKLIN,1506.0,1557
6,HAMPDEN,11629.0,11941
7,HAMPSHIRE,3354.0,3342
8,MIDDLESEX,29445.0,29901
9,NANTUCKET,949.0,1072


### The suffix parameter
By default, Pandas does not allow duplicate column names. Therefore, if the two dataframes have a column other than the key column that has the same name, Pandas automatically uses suffixes to distinguish them after the merge.

In [82]:
# create a df with the unemployment data of MA in Feb of 2022
unemp_feb_22 = pd.DataFrame({'COUNTY':
                               ['BARNSTABLE',
                                'BERKSHIRE',
                                'BRISTOL',
                                'DUKES',
                                'ESSEX',
                                'FRANKLIN',
                                'HAMPDEN',
                                'HAMPSHIRE',
                                'MIDDLESEX',
                                'NANTUCKET',
                                'NORFOLK',
                                'PLYMOUTH',
                                'SUFFOLK',
                                'WORCESTER'],
                               'Feb Unemployment':
                               [7952,
                                3366,
                                17777,
                                804,
                                19134,
                                1653,
                                12830,
                                3305,
                                29335,
                                1071,
                                14045,
                                14072,
                                17043,
                                20000]})
unemp_feb_22

,COUNTY,Feb Unemployment
0,BARNSTABLE,7952
1,BERKSHIRE,3366
2,BRISTOL,17777
3,DUKES,804
4,ESSEX,19134
5,FRANKLIN,1653
6,HAMPDEN,12830
7,HAMPSHIRE,3305
8,MIDDLESEX,29335
9,NANTUCKET,1071


In [83]:
# merge the two and see what happens
unemp_feb.merge(unemp_feb_22, on='COUNTY')

,COUNTY,Feb Unemployment_x,Feb Unemployment_y
0,BARNSTABLE,7855,7952
1,BERKSHIRE,3162,3366
2,BRISTOL,17112,17777
3,DUKES,772,804
4,ESSEX,18486,19134
5,FRANKLIN,1557,1653
6,HAMPDEN,11941,12830
7,HAMPSHIRE,3342,3305
8,MIDDLESEX,29901,29335
9,NANTUCKET,1072,1071


The suffix '_x' indicates that the column is from the left dataframe. The suffix '_y' indicates that the column is from the right dataframe. You can make the suffixes more descriptive by setting the the value of the parameter 'suffixes'. 

In [84]:
# make the suffixes more descriptive
unemp_feb.merge(unemp_feb_22, on='COUNTY', suffixes=[' 2023', ' 2022'])

,COUNTY,Feb Unemployment 2023,Feb Unemployment 2022
0,BARNSTABLE,7855,7952
1,BERKSHIRE,3162,3366
2,BRISTOL,17112,17777
3,DUKES,772,804
4,ESSEX,18486,19134
5,FRANKLIN,1557,1653
6,HAMPDEN,11941,12830
7,HAMPSHIRE,3342,3305
8,MIDDLESEX,29901,29335
9,NANTUCKET,1072,1071


### concat()

The `.concat()` method stitches two dataframes together along the row axis or the column axis. It is often used to combine two datasets to form a larger one for further processing.   

By default, the `concat()` method concatenates multiple dataframes along the row axis. 

In [85]:
# concatenate unemp_feb and unemp_feb_22
pd.concat([unemp_feb, unemp_feb_22])

,COUNTY,Feb Unemployment
0,BARNSTABLE,7855
1,BERKSHIRE,3162
2,BRISTOL,17112
3,DUKES,772
4,ESSEX,18486
5,FRANKLIN,1557
6,HAMPDEN,11941
7,HAMPSHIRE,3342
8,MIDDLESEX,29901
9,NANTUCKET,1072


You can see that the indexes from the original dataframes are preserved after the concatenation. If you would like to reset the index after concatenation, you can set the parameter `ignore_index` to `True`.

In [42]:
# set ignore_index to False
pd.concat([unemp_feb, unemp_feb_22], ignore_index=True)

,COUNTY,Feb Unemployment
0,BARNSTABLE,7855
1,BERKSHIRE,3162
2,BRISTOL,17112
3,DUKES,772
4,ESSEX,18486
5,FRANKLIN,1557
6,HAMPDEN,11941
7,HAMPSHIRE,3342
8,MIDDLESEX,29901
9,NANTUCKET,1072


The two dataframes `unemp_feb` and `unemp_feb_22` have the same columns. Both have a column `COUNTY` and a column `Feb Unemployment`. What if we have two dataframes that have non-matching columns? For example, the dataframe `unemp_jan` has a `COUNTY` column and a `Jan Unemployment` column. If we concatenate `unemp_feb` and `unemp_jan`, what will happen?

In [43]:
# concatenate two dfs that have non-matching columns
pd.concat([unemp_feb, unemp_jan])

,COUNTY,Feb Unemployment,Jan Unemployment
0,BARNSTABLE,7855.0,NaN
1,BERKSHIRE,3162.0,NaN
2,BRISTOL,17112.0,NaN
3,DUKES,772.0,NaN
4,ESSEX,18486.0,NaN
5,FRANKLIN,1557.0,NaN
6,HAMPDEN,11941.0,NaN
7,HAMPSHIRE,3342.0,NaN
8,MIDDLESEX,29901.0,NaN
9,NANTUCKET,1072.0,NaN


If you would like to concatenate two dataframes along the column axis, just set the value of the 'axis' parameter to 1. 

In [44]:
# concatenate along the column axis
pd.concat([unemp_jan.set_index('COUNTY'), unemp_feb.set_index('COUNTY')], axis=1)

,Jan Unemployment,Feb Unemployment
COUNTY,,
BARNSTABLE,7233,7855
BERKSHIRE,3078,3162
BRISTOL,15952,17112
DUKES,696,772
ESSEX,17573,18486
FRANKLIN,1506,1557
HAMPDEN,11629,11941
HAMPSHIRE,3354,3342
MIDDLESEX,29445,29901


## Duplicates and unique values

After we combine several dataframes into a big one, a common practice is to remove the duplicates in the big dataframe. 

### drop_duplicates()

There is a handy method in Pandas that can remove duplicates, i.e.  `drop_duplicates()`. 

In [86]:
# make a df with duplicates
rate = pd.DataFrame([['Pandas', 'Morning', 7], 
                     ['Pandas', 'Morning', 7],
                     ['Pandas', 'Evening', 8],
                     ['PythonBasics','Morning', 9]],
                    columns=['Course', 'Session', 'Rating'])
rate

,Course,Session,Rating
0,Pandas,Morning,7
1,Pandas,Morning,7
2,Pandas,Evening,8
3,PythonBasics,Morning,9


By default, `.drop_duplicates()` considers all columns when removing duplicates. 

In [87]:
# drop duplicates
rate.drop_duplicates()

,Course,Session,Rating
0,Pandas,Morning,7
2,Pandas,Evening,8
3,PythonBasics,Morning,9


You can specify the column(s) to look for duplicates by setting the `subset` parameter. 

In [88]:
# set the 'subset' parameter
rate.drop_duplicates(subset=['Course'])

,Course,Session,Rating
0,Pandas,Morning,7
3,PythonBasics,Morning,9


By default, the first occurrence of the duplicates will be kept. If you would like to keep the last occurrence, you can set the `keep` parameter to `last`. 

In [89]:
# set the 'keep' parameter to 'last'
rate.drop_duplicates(subset=['Course'], keep='last')

,Course,Session,Rating
2,Pandas,Evening,8
3,PythonBasics,Morning,9


To drop all duplicates, you can set the `keep` parameter to `False`.

In [90]:
# drop all duplicates
rate.drop_duplicates(subset=['Course'], keep=False)

,Course,Session,Rating
3,PythonBasics,Morning,9


After you remove all duplicates from a column, the values left in that column are unique. 

There is another useful method to find the unique values in a certain column in a dataframe is the `.unique()` method 

In [91]:
# Get the unique courses in df rate
rate['Course'].unique()

array(['Pandas', 'PythonBasics'], dtype=object)

## Vectorized operations in Pandas

As you have seen so far, a lot of the methods in Pandas can work on the values in a row or a column in a batch. For example, if you call `.isna()` on a column, it checks, for each value in that column, whether it is NaN or not. This kind of operation where values are taken and operated on in a batch is called **vectorized operations** in Pandas.

In [ ]:
# a simple example of vectorized operation
df = pd.DataFrame({'number':[1,2,3,4],
                  'year': [2020, 2021, 2022, 2023]})

# grab the 'number' column and add 2 to it
df['number'] + 2

As you can see, the addition is applied element-wise to the integers in the `number` column. How do we understand this operation? Why don't we need to write a for loop to iterate over the values in the `number` column and add 2 to each of them? 

Actually, this vectorized addition can be broken down into two steps. 
* First of all, the addend 2 undergoes a process called **broadcasting**. It is stretched into a vector of the same shape as the `number` column. 


<center><img src="./data/Pandas3_vectorized2.png" width="40"></center>


* Second, we add the integers in the `number` column and the vector of 2s in the same way that we add two vectors in mathematics. We have **vectorized** the addition in this case. 



<center><img src="./data/Pandas3_vector.png" width="200"></center>

When we add two vectors, the two numbers in the corresponding positions in the two vectors are added together.

### The importance of shape

A prerequisite for applying an operation to two vectors is to make sure that they have compatible shapes. In the previous example, we are able to add 2 to the `number` column in a vectorized way because the value 2 can be **broadcast** into a vector whose shape is compatible with the `number` column.

In [ ]:
# applying an operation to two vectors of incompatible shapes
# throws an error
df['number'] + [1, 2]

Note that because the vectorized operations apply element-wise, we do not need to write a for loop to iterate over the values in a column and repeat the desired operation to each value. 

Whenever you are tempted to write a for loop when processing data in Pandas, take a pause and think about whether there is a vectorized way to do it. In most cases, vectorized operations are **faster and more efficient**. 

In [ ]:
# create a df with 1000 rows of random ints between 1 and 1000
import numpy as np
df_test = pd.DataFrame({'a':np.random.randint(1,1000, 1000),
                    'b':np.random.randint(1,1000, 1000)})

Let's use a for loop to add two columns and get the execution time.

In [ ]:
%%timeit -n 10
for index, row in df_test.iterrows():
    row['sum'] = row['a'] + row['b']

Let's add two columns in a vectorized way and get the execution time. 

In [ ]:
%%timeit -n 10
df_test['sum'] = df_test['a'] + df_test['b']

### Some more examples of vectorized operations
With numerical data, it is really convenient to use vectorized operations. 

In [ ]:
# Get the unemployment rate in MA in feb 2023
unemp_feb['Feb Labor Force'] = [108646,
                                61320,
                                300028,
                                8403,
                                424616,
                                40085,
                                225735,
                                90085,
                                914953,
                                6837,
                                395358,
                                285373,
                                455084,
                                446151]
unemp_feb['Feb Unemployment Rate'] = (unemp_feb['Feb Unemployment']/
                                     unemp_feb['Feb Labor Force'])
unemp_feb

You have learned a bunch of string methods in [Python Basics](../Python-basics/python-basics-1.ipynb). Almost all Python's built-in string methods have counterparts in Pandas's vectorized string methods, e.g., `upper()`, `lower()`, `isalpha()`, `startswith()`, `split()`, so on and so forth. 

In [ ]:
# make all county names in lower case
unemp_jan['COUNTY'].str.lower()

Note that we will first use `.str` to access the values in a column as strings and then call a certain method. Let's see an example of another method `.split()`. 

In [ ]:
# create a df with some full names
full_name = bm_21['FullName'].copy().loc[:5]
full_name

In [ ]:
# Split the names into first names and last names
full_name.str.split()

With strings, a common operation is to search for a certain substring in them. You have seen an example in Pandas basics 2 where we extract all failed banks whose name contains the word 'Community'. 

In [ ]:
# a reminder of the contains() method
full_name.loc[full_name.str.contains('Leo')]

Another common operation with strings is to find out all strings that match a certain pattern. There are some string methods that accept regular expressions which can be used to specify patterns.   

In [ ]:
# search strings of a certain pattern
full_name.str.extract('(^L.*[nu]$)')

If you are interested in learning more about regular expressions, check out [this](../Regular-expressions/regular-expressions.ipynb) constellate notebook. 

## apply() and where()

There are a lot of methods in Pandas used for data processing. In this section, we focus our attention on two: `apply()` and `where()`. 

The `.apply()` method allows you to apply a self-defined function to rows and columns in a dataframe. 

Suppose you have a dataframe that contains the words produced by a baby each day. 

In [ ]:
# use apply to process data
vocab = pd.DataFrame([[{'duck', 'dog'}, {'dog', 'bird'}],
                    [{'hippo', 'sky'}, {'sky', 'cloud'}]],
                    columns=['day1', 'day2'],
                    index=['Alex', 'Ben'])
vocab

Now, suppose for each baby, you would like to extract the words that appear in both days so that you know which words were consistently produced them. 

In [ ]:
### use apply to get the intersection between sets in each row

# define a function that takes a row and returns the desired intersection
def get_common_words(r):
    return r['day1'] & r['day2'] 

# pass the function to .apply()
vocab.apply(get_common_words, axis=1)

If you are more comfortable with the lambda function, you can pass a lambda function directly to the `.apply()` method.

In [ ]:
# pass a lambda function to apply()
vocab.apply(lambda r: r['day1'] & r['day2'], axis=1)

The `lambda` function may look a little bit complicated to read. A good way to read this function is to break it down into two parts. What goes before the colon is the input to this function; what goes after the colon is the output of this function. 

The `.where()` method allows you to manipulate the data using the if-else logic. The syntax of the `.where()` method is `.where(condition, other)`. The values fulfilling the condition will be kept and the values that do not fulfill the condition are changed to the value specified by 'other'. 

Suppose you are a middle school teacher. You have a report of the grades for the most English test. 

In [ ]:
# create a df containing English grades
eng = pd.DataFrame({'grade': [90, 88, 70, 55]},
                  index=['Alice', 'Becky', 'Cindy', 'Dave'])
eng

And you would like to change any grade below 60 to 'F'. 

In [ ]:
# use .where() to change any grade below 60 to 'F'
eng.where(eng['grade']>=60, 'F')

You can do the same thing using the `.apply()` method. However, the if-else logic will need to be written into the function passed into `.apply()`.

In [ ]:
# use apply to change the grades below 60 to 'F'
eng['grade'].apply(lambda x: 'F' if x < 60 else x)

We have seen how to apply the if-else logic to the data in a dataframe. How about if-elif-else? Suppose you would like to change any grade 90 and above to 'A', any grade between 80-89 to 'B', 70-79 to 'C', 60-69 to 'D', and any grade below 60 to 'F'. 

In [ ]:
# change the number grades to letter grades
def convert_grade(x):
    if x >= 90:
        return 'A'
    elif 80<x<89:
        return 'B'
    elif 70<x<79:
        return 'C'
    elif 60<x<69:
        return 'D'
    else:
        return 'F'
eng['grade'].apply(convert_grade)